# 02 · Train Your First BC Policy / 亲自训练策略
预计 50–60 分钟用于理解、运行和比较；本机低维训练本身仅数秒。Policy（策略）把观测映射到动作；Training（训练）通过专家动作更新网络参数。默认是确定性 MLP（多层感知机），16 → 256 → 256 → 256 → 7，损失为动作均方误差 MSE。

In [ ]:
from pathlib import Path
import os, sys, json
from IPython.display import display, Markdown, Image, Video
from manipulation_lab.common import ROOT, DATA, write_json
from manipulation_lab.notebook import workspace, command, next_stage, stage_path
GROUP = os.environ.get("LAB_GROUP", "group_01")  # 教师为每组分配；只改自己的组
RUN = workspace(GROUP)
settings = json.loads((ROOT / "configs/bc_student.json").read_text())
settings

核心更新逻辑如下；真正训练由后面的独立脚本调用官方 robomimic BC。专家动作是标签，网络预测不参与产生这些标签。验证集由完整轨迹组成，用于选择 checkpoint；测试集不参与更新或模型选择。
```python
pred_action = policy(obs)
loss = criterion(pred_action, expert_action)
optimizer.zero_grad()
loss.backward()
optimizer.step()
```

In [ ]:
import h5py, torch
from manipulation_lab.common import policy_observation
from manipulation_lab.policy import make_config, make_model
from manipulation_lab.dataset import inspect
info = inspect(DATA)
config = make_config(settings)
example_model = make_model(config, info["policy_observation_shapes"], torch.device("cpu"))
print(example_model.nets["policy"])
with h5py.File(DATA, "r") as f:
    episode = info["splits"]["train"][0]
    print("Expert action labels (first 3):", f[f"data/{episode}/actions"][:3])
del example_model

In [ ]:
# 可修改学习率、训练轮数；保留同一划分和测试集以便比较。
settings["learning_rate"] = 0.001
settings["epochs"] = 50
settings["device"] = "cuda"  # 没有GPU时明确改为cpu
TRAINING = next_stage(RUN, "training")
CONFIG_PATH = RUN / (TRAINING.name + "_config.json")
write_json(CONFIG_PATH, settings)
command("train.py", "--config", CONFIG_PATH, "--output", TRAINING)
CHECKPOINT = TRAINING / "student_policy.pth"
assert CHECKPOINT.is_file()
print("Your checkpoint:", CHECKPOINT)

In [ ]:
display(Image(filename=str(TRAINING / "training_curve.png")))
summary = json.loads((TRAINING / "training_summary.json").read_text())
print({k:summary[k] for k in ["epochs", "best_epoch", "best_validation_mse", "elapsed_seconds", "checkpoint_reload_passed"]})

讨论：训练损失下降，验证损失一定下降吗？验证动作误差更小，就一定能完整抓取、搬运、释放并稳定放置吗？Notebook 03 将使用你刚训练的权重测试。